[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JMasr/ai-tutorials/blob/main/es/whisper-optuna-mlflow.ipynb)

Autor: José Manuel Ramírez Sánchez · https://jmramirez.engineer

# Fine-tuning de Whisper-tiny con Optuna, validación cruzada y MLflow

Este tutorial muestra un ciclo experimental completo y reproducible para ajustar `openai/whisper-tiny` con un fragmento pequeño de voz real, pública y en inglés (sin datos clínicos):

1. Carga 73 locuciones de LibriSpeech y bloquea un capítulo como `test` antes de cualquier otra cosa.
2. Mide el WER *zero-shot* del checkpoint sin tocar sobre `test`: la línea base que cualquier modelo ajustado debe superar.
3. Usa Optuna para buscar hiperparámetros mediante validación cruzada agrupada por capítulo.
4. Registra en MLflow los parámetros y métricas de cada *trial*, *fold* y refit.
5. Reentrena con todo `train` y los `best_params`, evalúa una sola vez sobre `test` y decide qué modelo conservar.

> **Objetivo didáctico.** El conjunto es deliberadamente diminuto (58 locuciones de entrenamiento de un único hablante): sirve para aprender el contrato del experimento (test bloqueado, folds agrupados, línea base registrada, decisión explícita), no para obtener un ASR mejor. En CPU, la configuración por defecto tarda unos 10 minutos; en GPU se puede aumentar `N_TRIALS` y el número de épocas.

No se registra audio, texto ni pesos del modelo en MLflow: solo metadatos, parámetros y métricas agregadas.

## 0. Requisitos y diseño experimental

Instala las dependencias:

```bash
pip install torch transformers==4.57.6 datasets accelerate optuna mlflow soundfile
```

El flujo aplica cuatro usos distintos de los datos:

| Fase | Datos usados | Decisión que permite |
|---|---|---|
| Línea base | `test` (capítulo 128104, 15 locuciones) | WER *zero-shot* del checkpoint sin tocar: la referencia a superar |
| HPO | `train` (58 locuciones), dejando un capítulo fuera: 2 folds | Seleccionar hiperparámetros por pérdida media de validación |
| Refit | `train` completo | Ajustar el candidato final con los hiperparámetros ganadores |
| Evaluación final | `test`, evaluado una vez | Comparar el WER del refit con la línea base y quedarse con el mejor modelo |

`test` se bloquea antes que nada y nunca se usa para elegir parámetros, épocas ni folds. Solo se puntúa dos veces, una para el checkpoint sin tocar y otra para el modelo reentrenado, y esos dos números se comparan, nunca se optimizan contra él.

In [1]:
import gc
import inspect
import io
import os
import random
import shutil
import string
import tempfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import mlflow
import numpy as np
import optuna
import soundfile as sf
import torch
from datasets import Audio, Dataset, load_dataset
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperConfig,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

SEED = 17
CHECKPOINT = 'openai/whisper-tiny'
DATASET_ID = 'hf-internal-testing/librispeech_asr_dummy'
DATASET_CONFIG = 'clean'
DATASET_SPLIT = 'validation'
TEST_CHAPTER = 128104  # Locked before anything else; never used for selection.
GROUP_KEY = 'chapter_id'  # With multi-speaker data, group by speaker (or patient) instead.
SAMPLE_RATE = 16_000
N_SPLITS = 2  # Leave-one-chapter-out over the two training chapters.
N_TRIALS = 3  # Total trial budget; raise this only after the end-to-end flow has been validated.
MAX_NEW_TOKENS = 225
EXPERIMENT_NAME = 'whisper-tiny-optuna-cv-tutorial'
OPTUNA_STUDY_NAME = 'whisper-tiny-librispeech-cv-hpo-v1'
# Set OPTUNA_STORAGE_URL outside the notebook to resume trials from a database.
# Example: sqlite:////absolute/path/optuna-whisper.db
OPTUNA_STORAGE_URL = os.environ.get('OPTUNA_STORAGE_URL')

# An explicit local URI makes the tutorial runnable without remote credentials.
# Export MLFLOW_TRACKING_URI before starting Jupyter to use a managed server.
default_tracking_uri = f'file:{Path.cwd() / "mlruns-whisper-optuna-tutorial"}'
TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', default_tracking_uri)
# MLflow >= 3.x rejects the file backend unless explicitly allowed; keep it for this local demo.
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def set_seed(seed: int) -> None:
    """Seed every random source used by this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'device: {DEVICE}')
print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Optuna persistent storage enabled: {OPTUNA_STORAGE_URL is not None}')

.../site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/09/29 00:24:03 INFO mlflow.tracking.fluent: Experiment with name 'whisper-tiny-optuna-cv-tutorial' does not exist. Creating a new experiment.


device: cpu
MLflow tracking URI: file:./mlruns-whisper-optuna-tutorial
Optuna persistent storage enabled: False


## 1. Voz real: un subconjunto de LibriSpeech

Los datos son `hf-internal-testing/librispeech_asr_dummy` (configuración `clean`, split `validation`): 73 locuciones de inglés leído de un único hablante (1272) repartidas en tres capítulos de libro, de 1,6 a 29,4 s, muestreadas a 16 kHz. Es un fragmento mínimo de **LibriSpeech** (V. Panayotov, G. Chen, D. Povey y S. Khudanpur, *LibriSpeech: an ASR corpus based on public domain audio books*, ICASSP 2015), distribuido bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Las transcripciones de referencia están en mayúsculas y sin puntuación; las pasamos a minúsculas para usarlas como etiquetas.

La columna de audio se carga con `decode=False` y los bytes FLAC se decodifican con `soundfile`, porque las versiones recientes de `datasets` necesitan el paquete adicional `torchcodec` para decodificar audio por sí mismas. Cada registro conserva `id`, `speaker_id` y `chapter_id` para que las particiones siguientes puedan agrupar por ellos.

In [2]:
def decode_flac(audio_field: dict) -> np.ndarray:
    """Decode the raw FLAC bytes stored in the dataset into a float32 waveform."""
    waveform, sample_rate = sf.read(io.BytesIO(audio_field['bytes']), dtype='float32')
    if sample_rate != SAMPLE_RATE:
        raise ValueError(f'Expected {SAMPLE_RATE} Hz audio, got {sample_rate} Hz')
    return waveform


# decode=False returns the encoded bytes, so soundfile decodes them and torchcodec is not needed.
raw_dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=DATASET_SPLIT)
raw_dataset = raw_dataset.cast_column('audio', Audio(decode=False))

records = [
    {
        'id': row['id'],
        'speaker_id': int(row['speaker_id']),
        'chapter_id': int(row['chapter_id']),
        'transcription': row['text'].lower(),
        'audio': decode_flac(row['audio']),
    }
    for row in raw_dataset
]

durations = np.array([len(record['audio']) / SAMPLE_RATE for record in records])
print(f'{len(records)} utterances | speakers: {sorted({r["speaker_id"] for r in records})}')
print(f'utterances per chapter: {dict(sorted(Counter(r["chapter_id"] for r in records).items()))}')
print(f'duration: min {durations.min():.1f} s | median {np.median(durations):.1f} s | max {durations.max():.1f} s')
print(f'total audio: {durations.sum() / 60:.1f} min')

73 utterances | speakers: [1272]
utterances per chapter: {128104: 15, 135031: 25, 141231: 33}
duration: min 1.6 s | median 5.6 s | max 29.4 s
total audio: 8.0 min


### 1.1 Un `test` bloqueado y folds agrupados por capítulo

El orden importa. Primero, el capítulo 128104 (15 locuciones) pasa a ser `test`, antes de que ningún otro paso mire los datos. Después, los folds de HPO se construyen con los otros dos capítulos (58 locuciones) dejando un capítulo fuera: cada fold entrena con un capítulo y valida con el otro.

¿Por qué agrupar? Las locuciones de una misma sesión de grabación comparten voz, micrófono, sala y vocabulario. Si se reparten al azar, cada clip de validación tiene casi-gemelos en entrenamiento y la pérdida de validación sale optimista. Con datos reales se agrupa por hablante, o por paciente en un corpus clínico, para que ninguna identidad aparezca a ambos lados de una partición. Este subconjunto tiene un único hablante, así que el capítulo es la única agrupación disponible; aun así, mantiene la sesión y el tema de cada capítulo fuera de su propio fold de validación. `make_grouped_folds` sustituye a la antigua función estratificada: cada grupo cae en exactamente un fold de validación, y una aserción comprueba que ningún grupo se filtra al entrenamiento.

In [3]:
# 1) Lock the test chapter first. Nothing below this line may look at its metrics until the end.
test_records = [record for record in records if record['chapter_id'] == TEST_CHAPTER]
train_records = [record for record in records if record['chapter_id'] != TEST_CHAPTER]


def make_grouped_folds(
    records: list[dict], group_key: str, n_splits: int
) -> list[tuple[list[dict], list[dict]]]:
    """Return train/validation folds where every group lands in exactly one validation fold."""
    group_sizes = Counter(record[group_key] for record in records)
    if len(group_sizes) < n_splits:
        raise ValueError(f'Need at least {n_splits} groups, found {len(group_sizes)}')

    # Greedy balancing: assign the largest remaining group to the smallest validation bucket.
    buckets: list[set] = [set() for _ in range(n_splits)]
    bucket_sizes = [0] * n_splits
    for group, size in sorted(group_sizes.items(), key=lambda item: (-item[1], item[0])):
        target = int(np.argmin(bucket_sizes))
        buckets[target].add(group)
        bucket_sizes[target] += size

    folds = []
    for validation_groups in buckets:
        fold_train = [r for r in records if r[group_key] not in validation_groups]
        fold_validation = [r for r in records if r[group_key] in validation_groups]
        assert not {r[group_key] for r in fold_train} & validation_groups
        folds.append((fold_train, fold_validation))
    return folds


# 2) Build the HPO folds from train only, grouped by chapter.
folds = make_grouped_folds(train_records, group_key=GROUP_KEY, n_splits=N_SPLITS)

print(f'train: {len(train_records)} utterances | test (chapter {TEST_CHAPTER}): {len(test_records)} utterances')
for index, (fold_train, fold_validation) in enumerate(folds):
    train_groups = sorted({r[GROUP_KEY] for r in fold_train})
    validation_groups = sorted({r[GROUP_KEY] for r in fold_validation})
    print(
        f'fold {index}: train={len(fold_train)} {train_groups} | '
        f'validation={len(fold_validation)} {validation_groups}'
    )

train: 58 utterances | test (chapter 128104): 15 utterances
fold 0: train=25 [135031] | validation=33 [141231]
fold 1: train=33 [141231] | validation=25 [135031]


## 2. Preparar Whisper y el *data collator*

Whisper recibe espectrogramas log-Mel y las etiquetas son IDs del tokenizer. El procesador se configura para transcripción en inglés (`language='English'`, `task='transcribe'`), de modo que cada etiqueta empieza con los mismos tokens de prompt que el modelo usa en inferencia. La máscara convierte el padding de las etiquetas en `-100`, que PyTorch ignora al calcular la pérdida.

In [4]:
processor = WhisperProcessor.from_pretrained(CHECKPOINT, language='English', task='transcribe')
model_config = WhisperConfig.from_pretrained(CHECKPOINT)


def preprocess_record(record: dict) -> dict:
    """Convert an in-memory waveform and text label to Whisper inputs."""
    input_features = processor.feature_extractor(
        record['audio'], sampling_rate=SAMPLE_RATE
    ).input_features[0]
    labels = processor.tokenizer(record['transcription']).input_ids
    return {'input_features': input_features, 'labels': labels}


def to_dataset(records: list[dict]) -> Dataset:
    """Preprocess records and drop IDs, waveforms and text before Trainer sees them."""
    raw = Dataset.from_list(records)
    return raw.map(preprocess_record, remove_columns=raw.column_names)


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor
    decoder_start_token_id: int

    def __call__(self, features: list[dict]) -> dict[str, torch.Tensor]:
        input_features = [{'input_features': feature['input_features']} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        label_features = [{'input_ids': feature['labels']} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor, decoder_start_token_id=model_config.decoder_start_token_id
)
print('Processor loaded. The model weights will be reloaded for every fold.')

Processor loaded. The model weights will be reloaded for every fold.


## 3. Utilidades de CV, entrenamiento y MLflow

Cada fold crea un modelo nuevo desde el mismo checkpoint. Esta es una condición esencial de la validación cruzada: reutilizar pesos ajustados de un fold contaminaría el siguiente. La métrica objetivo es `eval_loss` promedio: es densa, barata de calcular y no exige generar texto en cada trial. La generación se configura para transcripción en inglés (`language='english'`, `task='transcribe'`).

In [5]:
def load_fresh_model() -> WhisperForConditionalGeneration:
    """Load an unmodified Whisper checkpoint for an independent training run."""
    model = WhisperForConditionalGeneration.from_pretrained(CHECKPOINT)
    model.config.use_cache = False
    model.generation_config.language = 'english'
    model.generation_config.task = 'transcribe'
    # Language and task above replace the legacy forced_decoder_ids prompt.
    model.generation_config.forced_decoder_ids = None
    return model


def build_trainer(
    model: WhisperForConditionalGeneration,
    train_records: list[dict],
    validation_records: list[dict] | None,
    hyperparameters: dict[str, float | int],
    output_dir: Path,
    run_seed: int,
) -> Seq2SeqTrainer:
    """Build a no-checkpoint trainer; without validation records it never evaluates on its own."""
    training_argument_kwargs = {
        'output_dir': str(output_dir),
        'learning_rate': float(hyperparameters['learning_rate']),
        'num_train_epochs': float(hyperparameters['num_train_epochs']),
        'per_device_train_batch_size': 2,
        'per_device_eval_batch_size': 2,
        'save_strategy': 'no',
        'logging_strategy': 'no',
        'report_to': [],
        'fp16': False,
        'seed': run_seed,
        'data_seed': run_seed,
        'remove_unused_columns': True,
    }
    evaluation_argument = (
        'eval_strategy'
        if 'eval_strategy' in inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
        else 'evaluation_strategy'
    )
    training_argument_kwargs[evaluation_argument] = 'epoch' if validation_records else 'no'
    training_args = Seq2SeqTrainingArguments(**training_argument_kwargs)
    trainer_kwargs = {
        'model': model,
        'args': training_args,
        'train_dataset': to_dataset(train_records),
        'eval_dataset': to_dataset(validation_records) if validation_records else None,
        'data_collator': data_collator,
    }
    processing_argument = (
        'processing_class'
        if 'processing_class' in inspect.signature(Seq2SeqTrainer.__init__).parameters
        else 'tokenizer'
    )
    trainer_kwargs[processing_argument] = processor.feature_extractor
    return Seq2SeqTrainer(**trainer_kwargs)


def run_fold(
    fold_index: int,
    fold_train: list[dict],
    fold_validation: list[dict],
    hyperparameters: dict[str, float | int],
    run_seed: int,
) -> float:
    """Train one independent fold and return its validation loss."""
    set_seed(run_seed)
    model = load_fresh_model()
    temp_dir = Path(tempfile.mkdtemp(prefix=f'whisper-fold-{fold_index}-'))
    try:
        trainer = build_trainer(
            model, fold_train, fold_validation, hyperparameters, temp_dir, run_seed
        )
        training_output = trainer.train()
        evaluation = trainer.evaluate()
        validation_loss = float(evaluation['eval_loss'])
        mlflow.log_metrics(
            {
                'train_loss': float(training_output.training_loss),
                'eval_loss': validation_loss,
                'train_runtime_s': float(training_output.metrics['train_runtime']),
                'train_samples_per_second': float(training_output.metrics['train_samples_per_second']),
            }
        )
        return validation_loss
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        shutil.rmtree(temp_dir, ignore_errors=True)


def suggest_hyperparameters(trial: optuna.Trial) -> dict[str, float | int]:
    """Define a deliberately small, CPU-friendly search space."""
    return {
        'learning_rate': trial.suggest_float('learning_rate', 5e-6, 3e-5, log=True),
        'num_train_epochs': trial.suggest_int('num_train_epochs', 1, 2),
    }

### 3.1 Línea base *zero-shot* sobre `test`

Antes de entrenar nada, transcribimos `test` con el checkpoint sin tocar. Su WER es el número que un modelo ajustado debe superar, y se registra en el run padre de MLflow como `baseline_test_wer` para que la decisión final sea auditable.

El WER se calcula sobre texto normalizado, aplicado igual a referencias y predicciones. El notebook usa el normalizador de texto inglés de Whisper (`processor.tokenizer.normalize`), que pasa a minúsculas, elimina la puntuación y unifica grafías, números y abreviaturas; la celda siguiente comprueba que convierte "Mr." en "mister". Esto es necesario para una comparación justa: las referencias de LibriSpeech escriben "MISTER" mientras que Whisper escribe "Mr.", así que sin normalizar una simple convención ortográfica contaría como error para el modelo *zero-shot* pero no para un modelo ajustado con esas referencias. Si ese normalizador no está disponible, el código recurre a minúsculas y eliminación de puntuación, y registra cuál ha usado (`wer_normalizer`). El WER por distancia de edición no necesita dependencias adicionales.

In [6]:
_PUNCTUATION_TABLE = str.maketrans('', '', string.punctuation)


def simple_normalize(text: str) -> str:
    """Fallback normalizer: lowercase, strip punctuation, collapse whitespace."""
    return ' '.join(text.lower().translate(_PUNCTUATION_TABLE).split())


def normalize_text(text: str) -> str:
    """Prefer Whisper's English normalizer; fall back to the simple one if unavailable."""
    try:
        return processor.tokenizer.normalize(text)
    except (AttributeError, NotImplementedError, ValueError):
        return simple_normalize(text)


# Check which normalizer is active: Whisper's English normalizer spells out "Mr." as "mister".
NORMALIZER = (
    'whisper-english'
    if normalize_text('Mr. Quilter') == 'mister quilter'
    else 'lowercase-strip-punctuation'
)
print(f'normalizer: {NORMALIZER}')
print(f"'Mr. Quilter, the apostle.' -> {normalize_text('Mr. Quilter, the apostle.')!r}")


def word_error_rate(references: list[str], predictions: list[str]) -> float:
    """Compute corpus WER on normalized text without an extra notebook dependency."""
    def edit_distance(reference_words: list[str], predicted_words: list[str]) -> int:
        previous = list(range(len(predicted_words) + 1))
        for row_index, reference_word in enumerate(reference_words, start=1):
            current = [row_index]
            for column_index, predicted_word in enumerate(predicted_words, start=1):
                substitution = previous[column_index - 1] + (reference_word != predicted_word)
                insertion = current[column_index - 1] + 1
                deletion = previous[column_index] + 1
                current.append(min(substitution, insertion, deletion))
            previous = current
        return previous[-1]

    total_errors = 0
    total_reference_words = 0
    for reference, prediction in zip(references, predictions, strict=True):
        reference_words = normalize_text(reference).split()
        prediction_words = normalize_text(prediction).split()
        total_errors += edit_distance(reference_words, prediction_words)
        total_reference_words += len(reference_words)
    return total_errors / total_reference_words


def generate_transcriptions(
    model: WhisperForConditionalGeneration, records: list[dict], batch_size: int = 4
) -> list[str]:
    """Generate text only in memory; transcripts are never logged."""
    model.eval()
    predictions = []
    for start in range(0, len(records), batch_size):
        batch = records[start : start + batch_size]
        features = processor.feature_extractor(
            [record['audio'] for record in batch],
            sampling_rate=SAMPLE_RATE,
            return_tensors='pt',
        ).input_features.to(model.device)
        with torch.no_grad():
            generated_ids = model.generate(features, max_new_tokens=MAX_NEW_TOKENS)
        predictions.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
    return predictions

normalizer: whisper-english
'Mr. Quilter, the apostle.' -> 'mister quilter the apostle'


In [7]:
set_seed(SEED)
baseline_model = load_fresh_model()
baseline_predictions = generate_transcriptions(baseline_model, test_records)
baseline_test_wer = word_error_rate(
    [record['transcription'] for record in test_records], baseline_predictions
)
del baseline_model, baseline_predictions
gc.collect()
print(f'Zero-shot {CHECKPOINT} WER on test (chapter {TEST_CHAPTER}): {baseline_test_wer:.4f}')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Zero-shot openai/whisper-tiny WER on test (chapter 128104): 0.0971


## 4. HPO con Optuna y validación cruzada

La jerarquía de MLflow queda así: `tutorial` → `hpo` → `trial` → `fold`. El run `tutorial` guarda el diseño (`cv_group_key`, `test_chapter`, `wer_normalizer`) y `baseline_test_wer`; el `trial` guarda los hiperparámetros y la media/desviación de la pérdida; cada `fold` guarda sus métricas de entrenamiento y validación. Este nivel de detalle facilita distinguir un parámetro prometedor de un resultado dependiente de una sola partición. El espacio de búsqueda es pequeño a propósito: `learning_rate` entre 5e-6 y 3e-5 (escala logarítmica) y 1–2 épocas.

In [8]:
def run_hpo() -> tuple[optuna.Study, str]:
    """Optimize mean cross-validation loss and log the complete hierarchy."""
    sampler = optuna.samplers.TPESampler(seed=SEED)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=1)
    study = optuna.create_study(
        direction='minimize',
        sampler=sampler,
        pruner=pruner,
        study_name=OPTUNA_STUDY_NAME,
        storage=OPTUNA_STORAGE_URL,
        load_if_exists=OPTUNA_STORAGE_URL is not None,
    )

    with mlflow.start_run(run_name='whisper-tiny-optuna-cv-tutorial') as tutorial_run:
        tutorial_run_id = tutorial_run.info.run_id
        mlflow.set_tags(
            {
                'data.kind': 'public-speech',
                'data.source': f'{DATASET_ID}/{DATASET_CONFIG}/{DATASET_SPLIT}',
                'data.license': 'CC BY 4.0 (LibriSpeech)',
                'model.checkpoint': CHECKPOINT,
                'selection.metric': 'mean_cv_eval_loss',
                'privacy.note': 'No audio, transcript, or model artifact logged',
            }
        )
        mlflow.log_params(
            {
                'seed': SEED,
                'n_splits': N_SPLITS,
                'cv_group_key': GROUP_KEY,
                'test_chapter': TEST_CHAPTER,
                'target_trial_count': N_TRIALS,
                'optuna_pruner': pruner.__class__.__name__,
                'optuna_storage_enabled': OPTUNA_STORAGE_URL is not None,
                'train_examples': len(train_records),
                'test_examples': len(test_records),
                'wer_normalizer': NORMALIZER,
            }
        )
        # The zero-shot reference point, computed before any training.
        mlflow.log_metric('baseline_test_wer', float(baseline_test_wer))

        with mlflow.start_run(run_name='hpo', nested=True):
            def objective(trial: optuna.Trial) -> float:
                hyperparameters = suggest_hyperparameters(trial)
                fold_losses = []
                with mlflow.start_run(run_name=f'trial-{trial.number:03d}', nested=True):
                    mlflow.log_params({'trial_number': trial.number, **hyperparameters})
                    for fold_index, (fold_train, fold_validation) in enumerate(folds):
                        with mlflow.start_run(run_name=f'fold-{fold_index}', nested=True):
                            mlflow.log_params(
                                {
                                    'fold_index': fold_index,
                                    'fold_train_examples': len(fold_train),
                                    'fold_validation_examples': len(fold_validation),
                                    'fold_validation_groups': ','.join(
                                        str(g) for g in sorted({r[GROUP_KEY] for r in fold_validation})
                                    ),
                                    **hyperparameters,
                                }
                            )
                            fold_loss = run_fold(
                                fold_index,
                                fold_train,
                                fold_validation,
                                hyperparameters,
                                run_seed=SEED + trial.number * 100 + fold_index,
                            )
                            fold_losses.append(fold_loss)
                            partial_mean_loss = float(np.mean(fold_losses))
                            mlflow.log_metrics(
                                {
                                    'fold_eval_loss': fold_loss,
                                    'partial_mean_cv_eval_loss': partial_mean_loss,
                                },
                                step=fold_index + 1,
                            )
                        print(
                            f'trial {trial.number} fold {fold_index}: eval_loss={fold_loss:.4f} '
                            f'(partial mean {partial_mean_loss:.4f})'
                        )
                        trial.report(partial_mean_loss, step=fold_index + 1)
                        # Pruning after the last fold would save nothing, so only check before it.
                        is_last_fold = fold_index == len(folds) - 1
                        if not is_last_fold and trial.should_prune():
                            mlflow.set_tag('optuna.state', 'pruned')
                            raise optuna.TrialPruned(
                                f'Pruned after fold {fold_index} with mean loss {partial_mean_loss:.4f}'
                            )

                    mean_loss = float(np.mean(fold_losses))
                    std_loss = float(np.std(fold_losses))
                    mlflow.log_metrics(
                        {'mean_cv_eval_loss': mean_loss, 'std_cv_eval_loss': std_loss}
                    )
                    mlflow.set_tag('optuna.state', 'complete')
                    trial.set_user_attr('fold_eval_losses', fold_losses)
                    return mean_loss

            remaining_trials = max(0, N_TRIALS - len(study.trials))
            study.optimize(objective, n_trials=remaining_trials, gc_after_trial=True)
            mlflow.log_params({f'best_{key}': value for key, value in study.best_params.items()})
            mlflow.log_metric('best_mean_cv_eval_loss', float(study.best_value))

    return study, tutorial_run_id


optuna.logging.set_verbosity(optuna.logging.WARNING)
study, tutorial_run_id = run_hpo()
best_parameters = study.best_params
print('Best parameters:', best_parameters)
print(f'Best mean CV loss: {study.best_value:.4f}')
print(f'Parent MLflow run: {tutorial_run_id}')

.../site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss
1,No log,2.288820
2,No log,2.052194


trial 0 fold 0: eval_loss=2.0522 (partial mean 2.0522)


Epoch,Training Loss,Validation Loss
1,No log,2.148846
2,No log,1.744279


trial 0 fold 1: eval_loss=1.7443 (partial mean 1.8982)


...

Epoch,Training Loss,Validation Loss
1,No log,1.577891
2,No log,1.501230


trial 2 fold 0: eval_loss=1.5012 (partial mean 1.5012)


Epoch,Training Loss,Validation Loss
1,No log,1.554017
2,No log,1.494496


trial 2 fold 1: eval_loss=1.4945 (partial mean 1.4979)


Best parameters: {'learning_rate': 2.0481592635105394e-05, 'num_train_epochs': 2}
Best mean CV loss: 1.4979
Parent MLflow run: 686bb54ee5bb45a5b014855f1cb182f2


In [9]:
trial_summary = [
    {
        'trial': trial.number,
        'state': trial.state.name,
        'mean_cv_eval_loss': trial.value,
        **trial.params,
        'fold_losses': [round(loss, 4) for loss in trial.user_attrs.get('fold_eval_losses', [])],
    }
    for trial in study.trials
]

for row in sorted(trial_summary, key=lambda row: (row['mean_cv_eval_loss'] is None, row['mean_cv_eval_loss'])):
    print(row)

{'trial': 2, 'state': 'COMPLETE', 'mean_cv_eval_loss': 1.4978630542755127, 'learning_rate': 2.0481592635105394e-05, 'num_train_epochs': 2, 'fold_losses': [1.5012, 1.4945]}
{'trial': 0, 'state': 'COMPLETE', 'mean_cv_eval_loss': 1.898236334323883, 'learning_rate': 8.477424772958625e-06, 'num_train_epochs': 2, 'fold_losses': [2.0522, 1.7443]}
{'trial': 1, 'state': 'COMPLETE', 'mean_cv_eval_loss': 2.575037956237793, 'learning_rate': 7.046965481457174e-06, 'num_train_epochs': 1, 'fold_losses': [2.5585, 2.5916]}


### 4.1 Optuna Pruner: detener trials poco prometedores

Un *pruner* evita gastar cómputo en un trial cuyo resultado intermedio ya es peor que los trials previos. El notebook usa `MedianPruner`: después de `n_startup_trials=2` completos, compara la pérdida media parcial de cada trial con la mediana histórica en el mismo paso.

Aquí un *paso* es un fold: tras cada fold, `trial.report(partial_mean_loss, step=fold_index + 1)` comunica a Optuna la métrica intermedia, y `trial.should_prune()` decide si abortar el trial con `TrialPruned`. Al dejar un capítulo fuera, el paso 1 corresponde siempre al mismo capítulo de validación, así que las medias parciales son comparables entre trials. No se comprueba el pruning tras el último fold, porque detenerse ahí no ahorraría nada. MLflow conserva las métricas de los folds ya ejecutados y marca el run del trial como `optuna.state=pruned` (o `complete`).

> Con `N_TRIALS = 3` solo los trials posteriores a los dos primeros pueden podarse, y solo tras su primer fold. Para observar pruning, aumenta `N_TRIALS` (por ejemplo, a 10).

Consulta: [Optuna — pruners](https://optuna.readthedocs.io/en/stable/reference/pruners.html) y [Trial.report / Trial.should_prune](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html).

### 4.2 Persistir y reanudar trials con una base de datos

Por defecto, `OPTUNA_STORAGE_URL` es `None`: el estudio vive en memoria y se pierde al reiniciar el kernel. Para evitar recomputar trials ya terminados, configura una URI de storage **antes de iniciar Jupyter**. SQLite es suficiente para un único proceso; PostgreSQL o MySQL son preferibles para varios workers o ejecuciones concurrentes.

```bash
# Desarrollo local: ruta absoluta para no mezclar la BD con artefactos versionables
export OPTUNA_STORAGE_URL='sqlite:////tmp/optuna-whisper-tutorial.db'

# Servidor compartido: la URI debe llegar desde un gestor de secretos, nunca del notebook
export OPTUNA_STORAGE_URL="$OPTUNA_DATABASE_URL"
```

`study_name=OPTUNA_STUDY_NAME` y `load_if_exists=True` recuperan el mismo estudio. Además, `remaining_trials = N_TRIALS - len(study.trials)` trata `N_TRIALS` como presupuesto total: al reejecutar la celda, solo lanza los trials que falten. La URI no se loguea en MLflow, ya que una URI de servidor puede contener credenciales.

Si un proceso se interrumpe, revisa los trials que queden en estado `RUNNING` antes de relanzar workers; en producción, configura además el mecanismo de *heartbeat* de Optuna. Consulta: [Optuna — RDB storage](https://optuna.readthedocs.io/en/stable/tutorial/20_recipes/001_rdb.html) y [Optuna — distributed optimization](https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/004_distributed.html).

## 5. Refit final con `train` y una única evaluación sobre `test`

La selección ya ha terminado. Cargamos otra vez el checkpoint inicial y lo ajustamos con las 58 locuciones de `train` usando exactamente los hiperparámetros ganadores. `test` no se añade al refit: son los únicos datos que quedan para saber si el ajuste ha servido de algo. El trainer no evalúa durante el entrenamiento; al terminar, `test` se puntúa una sola vez (pérdida y WER). El run `refit-train` se registra como hijo del run principal con `refit_test_loss` y `refit_test_wer`.

In [10]:
def refit_with_best_parameters() -> dict[str, float]:
    """Refit from the base checkpoint on all of train, then evaluate once on test."""
    set_seed(SEED)
    model = load_fresh_model()
    temp_dir = Path(tempfile.mkdtemp(prefix='whisper-refit-'))
    try:
        trainer = build_trainer(
            model=model,
            train_records=train_records,
            validation_records=None,  # No evaluation during training: test is touched once, below.
            hyperparameters=best_parameters,
            output_dir=temp_dir,
            run_seed=SEED,
        )
        training_output = trainer.train()
        test_eval = trainer.evaluate(eval_dataset=to_dataset(test_records))
        predictions = generate_transcriptions(model, test_records)
        test_wer = word_error_rate(
            [record['transcription'] for record in test_records], predictions
        )
        return {
            'refit_train_loss': float(training_output.training_loss),
            'refit_test_loss': float(test_eval['eval_loss']),
            'refit_test_wer': float(test_wer),
            'refit_train_runtime_s': float(training_output.metrics['train_runtime']),
        }
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        shutil.rmtree(temp_dir, ignore_errors=True)


# Reopen the completed parent run so MLflow can attach a nested refit run.
with mlflow.start_run(run_id=tutorial_run_id):
    with mlflow.start_run(run_name='refit-train', nested=True):
        mlflow.log_params(
            {
                'refit_examples': len(train_records),
                'test_examples': len(test_records),
                **{f'best_{key}': value for key, value in best_parameters.items()},
            }
        )
        final_metrics = refit_with_best_parameters()
        mlflow.log_metrics(final_metrics)

print('Final aggregate metrics:', {key: round(value, 4) for key, value in final_metrics.items()})

Step,Training Loss


Final aggregate metrics: {'refit_train_loss': 0.5817, 'refit_test_loss': 0.5407, 'refit_test_wer': 0.1147, 'refit_train_runtime_s': 89.0798}


### 5.1 Decisión: ¿modelo ajustado o modelo base?

La regla se fija antes de mirar los números: el modelo ajustado solo se conserva si reduce el WER sobre `test` respecto a la línea base *zero-shot*. El resultado se registra como la etiqueta `decision` (`use-finetuned` o `keep-base`) en el run padre, junto a `baseline_test_wer` y `test_wer_delta`.

In [11]:
refit_test_wer = final_metrics['refit_test_wer']
decision = 'use-finetuned' if refit_test_wer < baseline_test_wer else 'keep-base'

with mlflow.start_run(run_id=tutorial_run_id):
    mlflow.set_tag('decision', decision)
    mlflow.log_metric('test_wer_delta', float(refit_test_wer - baseline_test_wer))

print(f'baseline (zero-shot) test WER: {baseline_test_wer:.4f}')
print(f'fine-tuned (refit)   test WER: {refit_test_wer:.4f}')
print(f'decision: {decision}')

baseline (zero-shot) test WER: 0.0971
fine-tuned (refit)   test WER: 0.1147
decision: keep-base


### 5.2 Resultado de esta ejecución

El fine-tuning **no** ha superado al modelo base. El checkpoint sin tocar obtiene un WER de 0,097 sobre `test`; el modelo reentrenado con las 58 locuciones de `train` y los mejores hiperparámetros (`learning_rate` ≈ 2,0e-5, 2 épocas) obtiene 0,115. La decisión es, por tanto, `keep-base`.

Esta es la lección del notebook, no un error. `whisper-tiny` se preentrenó con 680.000 horas de voz; 58 locuciones (menos de 6 minutos) de un único hablante no bastan para que un fine-tuning completo mejore eso, y actualizar todos los pesos con tan pocos datos sobre todo aleja al modelo de lo que ya hacía bien. El trabajo del pipeline es detectar precisamente esta situación, y por eso la línea base se mide antes de la búsqueda, se registra junto al refit y se convierte en una decisión explícita. Los siguientes pasos razonables son más datos (y más variados), una adaptación eficiente en parámetros como LoRA (consulta [PEFT y LoRA en Whisper](whisper-peft-lora.ipynb)) o adaptar solo el decoder con el encoder congelado.

## 6. Consultar los resultados registrados

La celda siguiente busca todos los runs del experimento y conserva los que pertenecen a este run del tutorial: la línea base guardada en el run padre, cada run `trial-*` bajo su hijo `hpo` (los runs de fold están un nivel más abajo) y el run `refit-train`.

Con backend local, se puede abrir `mlflow ui --backend-store-uri ./mlruns-whisper-optuna-tutorial` desde el directorio en el que se ejecutó el notebook. Con un tracking server remoto, los mismos runs aparecen en la interfaz de ese servidor.

In [12]:
import pandas as pd

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
all_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], output_format='pandas')


def column(frame: pd.DataFrame, name: str) -> pd.Series:
    """Return a column, or an empty one if MLflow never logged that key."""
    return frame[name] if name in frame.columns else pd.Series([None] * len(frame), index=frame.index)


parent = all_runs[all_runs['run_id'] == tutorial_run_id]
children = all_runs[column(all_runs, 'tags.mlflow.parentRunId') == tutorial_run_id]
hpo_run_ids = children.loc[children['tags.mlflow.runName'] == 'hpo', 'run_id'].tolist()
trials = all_runs[
    column(all_runs, 'tags.mlflow.parentRunId').isin(hpo_run_ids)
    & all_runs['tags.mlflow.runName'].str.startswith('trial-')
]
refit = children[children['tags.mlflow.runName'] == 'refit-train']

summary = pd.concat(
    [
        pd.DataFrame(
            {
                'run': 'baseline (zero-shot)',
                'learning_rate': None,
                'num_train_epochs': None,
                'mean_cv_eval_loss': None,
                'state': 'decision=' + column(parent, 'tags.decision').astype(str),
                'test_wer': column(parent, 'metrics.baseline_test_wer'),
            }
        ),
        pd.DataFrame(
            {
                'run': trials['tags.mlflow.runName'],
                'learning_rate': column(trials, 'params.learning_rate').astype(float),
                'num_train_epochs': column(trials, 'params.num_train_epochs'),
                'mean_cv_eval_loss': column(trials, 'metrics.mean_cv_eval_loss'),
                'state': column(trials, 'tags.optuna.state').fillna(trials['status']),
                'test_wer': None,
            }
        ).sort_values('run'),
        pd.DataFrame(
            {
                'run': refit['tags.mlflow.runName'],
                'learning_rate': column(refit, 'params.best_learning_rate').astype(float),
                'num_train_epochs': column(refit, 'params.best_num_train_epochs'),
                'mean_cv_eval_loss': None,
                'state': refit['status'],
                'test_wer': column(refit, 'metrics.refit_test_wer'),
            }
        ),
    ],
    ignore_index=True,
)
summary['learning_rate'] = summary['learning_rate'].map(lambda v: f'{v:.2e}' if pd.notna(v) else '')
summary.round(4).fillna('')

,run,learning_rate,num_train_epochs,mean_cv_eval_loss,state,test_wer
0,baseline (zero-shot),,,,decision=keep-base,0.097059
1,trial-000,8.48e-06,2,1.898236,complete,
2,trial-001,7.05e-06,1,2.575038,complete,
3,trial-002,2.05e-05,2,1.497863,complete,
4,refit-train,2.05e-05,2,,FINISHED,0.114706


## 7. Qué conservar al pasar a tus propios datos

- Sustituye únicamente la sección de carga de datos por el cargador canónico y conserva IDs de hablante/paciente/sesión para que los folds se agrupen y no haya fuga de identidad entre entrenamiento y validación.
- Reserva `test` antes de HPO; no elijas parámetros ni épocas con sus métricas.
- Registra la línea base *zero-shot* en el mismo experimento. Un modelo ajustado solo merece desplegarse si supera ese número sobre `test` bloqueado, y el pipeline tiene que poder concluir "quédate con el modelo base".
- Registra revisiones fijas del dataset y checkpoint, hashes de manifests y la configuración resuelta (p. ej. Hydra); nunca audio, transcripciones ni PHI en MLflow.
- Aumenta folds, trials y espacio de búsqueda en GPU. Dos folds y unos pocos trials aquí solo verifican el flujo completo.
- Para una ejecución remota, configura `MLFLOW_TRACKING_URI` y las credenciales fuera del notebook. No se deben persistir tokens ni secretos.
- Con pocos datos, prefiere una adaptación eficiente en parámetros al fine-tuning completo: consulta [PEFT y LoRA en Whisper](whisper-peft-lora.ipynb).

**Referencias:** [Optuna](https://optuna.readthedocs.io/), [MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking/), [Whisper en Transformers](https://huggingface.co/docs/transformers/model_doc/whisper), [LibriSpeech (OpenSLR 12)](https://www.openslr.org/12).